CLI-style report tool: picking a report type and date range, getting a summary vs. the previous period.

In [0]:
import pandas as pd
import sqlite3

BASE_DIR="/Volumes/assignment8/assignment8schema/assignment8volume"
DB_PATH = "/Volumes/assignment8/assignment8schema/assignment8volume/ecommerce.db"
conn=sqlite3.connect(DB_PATH)

In [0]:
print("Available Reports")
print("1. Daily")
print("2. Weekly")
print("3. Monthly")

choice = input("Enter your choice (1/2/3): ")
print(pd.read_sql("""
SELECT
MIN(DATE(order_date)) AS Start_Date,
MAX(DATE(order_date)) AS End_Date
FROM orders;
""", conn))

start_date = input("Enter Start Date (YYYY-MM-DD): ")
end_date = input("Enter End Date (YYYY-MM-DD): ")

if choice == "1":
    report_type = "Daily"
elif choice == "2":
    report_type = "Weekly"
elif choice == "3":
    report_type = "Monthly"
else:
    print("Invalid Choice")
    conn.close()
    exit()

query = """
SELECT
    COUNT(DISTINCT o.order_id) AS Total_Orders,
    ROUND(SUM(oi.quantity * oi.unit_price *
        (1 - oi.discount_percent/100.0)),2) AS Revenue,
    COUNT(DISTINCT o.customer_id) AS Unique_Customers
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
WHERE DATE(o.order_date)
BETWEEN ? AND ?;
"""

summary = pd.read_sql(query, conn, params=(start_date, end_date))

print("\n========== {} REPORT ==========".format(report_type))
print(summary)

print("\nTop 3 Products")

top_products = """
SELECT
    p.product_name,
    ROUND(SUM(oi.quantity*oi.unit_price*
    (1-oi.discount_percent/100.0)),2) AS Revenue
FROM order_items oi
JOIN orders o
ON oi.order_id=o.order_id
JOIN products p
ON oi.product_id=p.product_id
WHERE DATE(o.order_date)
BETWEEN ? AND ?
GROUP BY p.product_name
ORDER BY Revenue DESC
LIMIT 3;
"""

print(pd.read_sql(top_products, conn, params=(start_date, end_date)))

conn.close()

Available Reports
1. Daily
2. Weekly
3. Monthly


Enter your choice (1/2/3):  1

   Start_Date    End_Date
0  2024-01-01  2026-06-30


Enter Start Date (YYYY-MM-DD):  2024-02-02

Enter End Date (YYYY-MM-DD):  2025-02-02


========== Daily REPORT ==========
   Total_Orders      Revenue  Unique_Customers
0          1125  25076939.18               482

Top 3 Products
         product_name    Revenue
0    Positive Reality  509685.01
1    Staff Laptop Ten  411929.72
2  Wide Comic Believe  406530.24
